# Product-Issue Visualization

Takes the issue table produced by `01_launch_news_trend_analysis` and turns it into a
chart pack: articles by issue type, top brands, product categories, USPs, a daily issue
trend, a brand x issue-type heatmap, top issues and top keywords. Exports a summary
workbook and zips the charts. Missing values are labelled `Unknown`, never guessed.

> See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).

In [ ]:
import os, re, zipfile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from google.colab import files
    INPUT = list(files.upload().keys())[0]
except ImportError:
    INPUT = "../sample-data/news_issues.xlsx"   # synthetic sample

SHEET = "제품분석"   # sheet written by notebook 01
# Notebook 01 writes Korean column names; map them to English working names here.
COLS = {"대표이슈": "representative_title", "브랜드": "brand", "제품/제품명": "product",
        "제품카테고리": "category", "USP_표준": "usp", "이슈유형": "issue_type",
        "핵심어": "keywords", "기사수": "article_count", "최근기사일": "latest_article_date"}
df = pd.read_excel(INPUT, sheet_name=SHEET).rename(columns=COLS)
print(f"{len(df):,} issues, columns: {list(df.columns)}")

TEXT = ["representative_title", "brand", "product", "category", "usp", "issue_type", "keywords"]
a = df.copy()
for c in TEXT:
    if c in a:
        a[c] = a[c].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip().replace("", "Unknown")
a["articles"] = pd.to_numeric(a.get("article_count"), errors="coerce")
a["latest_date"] = pd.to_datetime(a.get("latest_article_date"), errors="coerce")

OUT = "issue_charts"; os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid")


def barh(series, title, fname, top=15):
    s = series.sort_values(ascending=False).head(top).sort_values()
    if s.empty:
        return
    plt.figure(figsize=(10, max(4, 0.45 * len(s))))
    s.plot(kind="barh"); plt.title(title); plt.xlabel("Articles"); plt.tight_layout()
    plt.savefig(f"{OUT}/{fname}", dpi=200, bbox_inches="tight"); plt.close()


known = lambda col: a[(a[col] != "Unknown") & a["articles"].notna()] if col in a else a.iloc[0:0]
barh(a.groupby("issue_type")["articles"].sum(), "Articles by issue type", "01_issue_type.png")
barh(known("brand").groupby("brand")["articles"].sum(), "Top 15 brands by articles", "02_brands.png")
barh(known("category").groupby("category")["articles"].sum(), "Articles by category", "03_categories.png")
barh(known("usp").groupby("usp")["articles"].sum(), "Top 15 USPs by articles", "04_usps.png")

if a["latest_date"].notna().any():
    daily = a.dropna(subset=["latest_date"]).groupby("latest_date")["articles"].sum().sort_index()
    plt.figure(figsize=(12, 5)); daily.plot(marker="o"); plt.title("Daily issue trend")
    plt.ylabel("Articles"); plt.tight_layout(); plt.savefig(f"{OUT}/05_daily_trend.png", dpi=200); plt.close()

if {"brand", "issue_type"} <= set(a.columns):
    t = known("brand"); t = t[t["issue_type"] != "Unknown"]
    pv = t.pivot_table(index="brand", columns="issue_type", values="articles", aggfunc="sum", fill_value=0)
    pv = pv.loc[pv.sum(axis=1).sort_values(ascending=False).head(15).index]
    if len(pv):
        plt.figure(figsize=(max(10, 1.2 * pv.shape[1]), 8))
        sns.heatmap(pv, annot=True, fmt=".0f"); plt.title("Brand x issue type")
        plt.tight_layout(); plt.savefig(f"{OUT}/06_brand_issue_heatmap.png", dpi=200); plt.close()

if "representative_title" in a:
    t = known("representative_title").nlargest(15, "articles")
    barh(t.set_index(t["representative_title"].str.slice(0, 45))["articles"], "Top 15 issues", "07_top_issues.png")

if "keywords" in a:
    kw = (a["keywords"][a["keywords"] != "Unknown"].str.split(r"[,;/|\n]+").explode()
          .str.strip().loc[lambda s: s != ""].value_counts())
    barh(kw, "Top 20 keywords", "08_keywords.png", top=20)

summary = f"{OUT}/issue_summary.xlsx"
with pd.ExcelWriter(summary, engine="openpyxl") as w:
    df.to_excel(w, sheet_name="original", index=False)
    a.to_excel(w, sheet_name="analysis", index=False)
    for col in ("issue_type", "brand", "category", "usp"):
        if col in a:
            (a[a[col] != "Unknown"].groupby(col)["articles"].sum().reset_index()
             .sort_values("articles", ascending=False).to_excel(w, sheet_name=f"by_{col}", index=False))

with zipfile.ZipFile("issue_charts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        z.write(f"{OUT}/{f}", arcname=f)
print("files:", sorted(os.listdir(OUT)))